## 导入库

In [1]:
import os
import json
from py2neo import Graph, Node

import pandas as pd

## 导入文件

In [2]:
df = pd.read_excel('yuanshen.xlsx')
print(len(df))
df.head()

62


,ID,城市,称号,性别,up,元素属性,武器类型,命之座,特殊料理,标签,介绍
0,琴,蒙德,蒲公英骑士,女,常驻UP,风元素,单手剑武器使用,幼狮座,提神醒脑披萨,治疗、控制、聚怪、减抗、攻速提升、移速提升、伤害减免、元素附着、烹饪、能量恢复,正直严谨的蒲公英骑士，蒙德西风骑士团的代理团长。
1,迪卢克,蒙德,晨曦的暗面,男,常驻UP,火元素,双手剑武器使用,夜枭座,蒙德往事」,锻造返还、自身附魔、自身伤害加成、自身攻速提升,坐拥蒙德大半酒业的贵公子，财力、人望、能力都令人无法小觑。
2,温迪,蒙德,风色诗人,男,限定UP,风元素,弓武器使用,歌仙座,风神杂烩菜,控制、聚怪、减抗、能量回复、滑翔消耗减少、自身伤害提升,蒙德城诸多吟游诗人中的一位，自由自在地穿行在街头巷尾
3,可莉,蒙德,逃跑的太阳,女,限定UP,火元素,法器武器使用,四叶草座,鱼香吐司,减防、能量回复、伤害提升、特产探索、自身体力消耗降低,西风骑士团禁闭室的常客，蒙德的爆破大师。人称「逃跑的太阳」。
4,魈,璃月,护法夜叉,男,限定UP,风元素,长柄武器武器使用,金翅鹏王座,美梦」,位移、攀爬消耗减少、元素转化、自身伤害提升、下落免疫,守护璃月的仙人，「夜叉」。美号「降魔大圣」，妙称「护法夜叉大将」。


## 创建实体节点

In [3]:
ID = []
city = []
element = []
weapon = []
cook = []

for each in df['ID']:
    ID.extend(each.split(','))
ID = set(ID)

for each in df['城市']:
    city.extend(each.split(','))
city = set(city)

for each in df['元素属性']:
    element.extend(each.split(','))
element = set(element)

for each in df['武器类型']:
    weapon.extend(each.split(','))
weapon = set(weapon)

for each in df['特殊料理']:
    cook.extend(each.split(','))
cook = set(cook)

### 角色字典信息

In [22]:
infos = [] # 疾病信息
for idx, row in df.iterrows():
    infos.append(dict(row))
dict(row).keys()

dict_keys(['ID', '城市', '称号', '性别', 'up', '元素属性', '武器类型', '命之座', '特殊料理', '标签', '介绍'])

In [5]:
infos

[{'ID': '琴',
  '城市': '蒙德',
  '称号': '蒲公英骑士',
  '性别': '女',
  'up': '常驻UP',
  '元素属性': '风元素',
  '武器类型': '单手剑武器使用',
  '命之座': '幼狮座',
  '特殊料理': '提神醒脑披萨',
  '标签': '治疗、控制、聚怪、减抗、攻速提升、移速提升、伤害减免、元素附着、烹饪、能量恢复',
  '介绍': '正直严谨的蒲公英骑士，蒙德西风骑士团的代理团长。'},
 {'ID': '迪卢克',
  '城市': '蒙德',
  '称号': '晨曦的暗面',
  '性别': '男',
  'up': '常驻UP',
  '元素属性': '火元素',
  '武器类型': '双手剑武器使用',
  '命之座': '夜枭座',
  '特殊料理': '蒙德往事」',
  '标签': '锻造返还、自身附魔、自身伤害加成、自身攻速提升',
  '介绍': '坐拥蒙德大半酒业的贵公子，财力、人望、能力都令人无法小觑。'},
 {'ID': '温迪',
  '城市': '蒙德',
  '称号': '风色诗人',
  '性别': '男',
  'up': '限定UP',
  '元素属性': '风元素',
  '武器类型': '弓武器使用',
  '命之座': '歌仙座',
  '特殊料理': '风神杂烩菜',
  '标签': '控制、聚怪、减抗、能量回复、滑翔消耗减少、自身伤害提升',
  '介绍': '蒙德城诸多吟游诗人中的一位，自由自在地穿行在街头巷尾'},
 {'ID': '可莉',
  '城市': '蒙德',
  '称号': '逃跑的太阳',
  '性别': '女',
  'up': '限定UP',
  '元素属性': '火元素',
  '武器类型': '法器武器使用',
  '命之座': '四叶草座',
  '特殊料理': '鱼香吐司',
  '标签': '减防、能量回复、伤害提升、特产探索、自身体力消耗降低',
  '介绍': '西风骑士团禁闭室的常客，蒙德的爆破大师。人称「逃跑的太阳」。'},
 {'ID': '魈',
  '城市': '璃月',
  '称号': '护法夜叉',
  '性别': '男',
  'up': '限定UP',
  '元素属性': '风元素',
  

## 创建关系边

In [6]:
def deduplicate(rels_old):
    '''关系去重函数'''
    rels_new = []
    for each in rels_old:
        if each not in rels_new:
            rels_new.append(each)
    return rels_new

### 关系：ID-城市

In [7]:
rels_city = []
for idx, row in df.iterrows():
    for each in row['城市'].split(','):
        rels_city.append([row['ID'], each])
rels_city = deduplicate(rels_city)

### ID-元素属性

In [8]:
rels_element = []
for idx, row in df.iterrows():
    for each in row['元素属性'].split(','):
        rels_element.append([row['ID'], each])
rels_element = deduplicate(rels_element)

### ID-武器类型

In [9]:
rels_weapon = []
for idx, row in df.iterrows():
    for each in row['武器类型'].split(','):
        rels_weapon.append([row['ID'], each])
rels_weapon = deduplicate(rels_weapon)

### ID-特殊料理

In [10]:
rels_cook = []
for idx, row in df.iterrows():
    for each in row['特殊料理'].split(','):
        rels_cook.append([row['ID'], each])
rels_cook = deduplicate(rels_cook)

### 城市-元素属性

In [11]:
city_element = []
for idx, row in df.iterrows():
    for each in row['元素属性'].split(','):
        city_element.append([row['城市'], each])
city_element = deduplicate(city_element)

## 连接图数据库

In [12]:
# 注意，这里的用户名为neo4j全局用户名，而非DBMS或者database的名称
g = Graph('http://localhost:7474', auth=('neo4j', '123456789'),name='neo4j')
g

Graph('http://localhost:7474', name='neo4j')

# 创建知识图谱实体

In [13]:
# 创建实体
node = Node('ID', name='角色', easy_get='旅行者')
g.create(node)

In [14]:
# 删除所有实体和关系
cypher = 'MATCH (n) DETACH DELETE n'
g.run(cypher)

(No data)

### 创建角色实体

In [15]:
count = 0
for _dict in infos:
    try:
        node = Node("角色",
                    name=_dict['ID'],
                    title=_dict['称号'],
                    sex=_dict['性别'],
                    up=_dict['up'],
                    constellation=_dict['命之座'],
                    tag=_dict['标签'],
                    intro = _dict['介绍'])
        g.create(node)
        count += 1
        print('创建角色实体：', _dict['ID'])
    except:
        pass
print('共创建 {} 个角色实体'.format(count))

创建角色实体： 琴
创建角色实体： 迪卢克
创建角色实体： 温迪
创建角色实体： 可莉
创建角色实体： 魈
创建角色实体： 旅行者
创建角色实体： 派蒙
创建角色实体： 莫娜
创建角色实体： 刻晴
创建角色实体： 七七
创建角色实体： 钟离
创建角色实体： 达达利亚
创建角色实体： 阿贝多
创建角色实体： 甘雨
创建角色实体： 神里绫华
创建角色实体： 胡桃
创建角色实体： 雷电将军
创建角色实体： 优菈
创建角色实体： 宵宫
创建角色实体： 枫原万叶
创建角色实体： 八重神子
创建角色实体： 珊瑚宫心海
创建角色实体： 埃洛伊
创建角色实体： 神里绫人
创建角色实体： 荒泷一斗
创建角色实体： 赛诺
创建角色实体： 申鹤
创建角色实体： 夜兰
创建角色实体： 提纳里
创建角色实体： 妮露
创建角色实体： 纳西妲
创建角色实体： 流浪者
创建角色实体： 北斗
创建角色实体： 安柏
创建角色实体： 丽莎
创建角色实体： 凯亚
创建角色实体： 芭芭拉
创建角色实体： 雷泽
创建角色实体： 行秋
创建角色实体： 菲谢尔
创建角色实体： 诺艾尔
创建角色实体： 班尼特
创建角色实体： 重云
创建角色实体： 香菱
创建角色实体： 凝光
创建角色实体： 砂糖
创建角色实体： 迪奥娜
创建角色实体： 辛焱
创建角色实体： 罗莎莉亚
创建角色实体： 云堇
创建角色实体： 烟绯
创建角色实体： 早柚
创建角色实体： 五郎
创建角色实体： 九条裟罗
创建角色实体： 托马
创建角色实体： 久岐忍
创建角色实体： 鹿野院平藏
创建角色实体： 柯莱
创建角色实体： 多莉
创建角色实体： 坎蒂丝
创建角色实体： 莱依拉
创建角色实体： 珐露珊
共创建 62 个角色实体


### 创建城市实体

In [16]:
for each in city:
    node = Node('City', name=each)
    g.create(node)
    print('创建实体 {}'.format(each))

创建实体 稻妻
创建实体 其它
创建实体 蒙德
创建实体 须弥
创建实体 璃月
创建实体 至冬


### 创建元素属性实体

In [17]:
for each in element:
    node = Node('Element', name=each)
    g.create(node)
    print('创建实体 {}'.format(each))

创建实体 火元素
创建实体 雷元素
创建实体 无、风、岩、雷、草元素
创建实体 岩元素
创建实体 风元素
创建实体 草元素
创建实体 Null
创建实体 冰元素
创建实体 水元素


### 创建武器类型实体

In [18]:
for each in weapon:
    node = Node('Weapon', name=each)
    g.create(node)
    print('创建实体 {}'.format(each))

创建实体 弓武器使用
创建实体 法器武器使用
创建实体 单手剑武器使用
创建实体 Null
创建实体 双手剑武器使用
创建实体 长柄武器武器使用


### 创建特殊料理

In [19]:
for each in cook:
    node = Node('Cook', name=each)
    g.create(node)
    print('创建实体 {}'.format(each))

创建实体 绝对不是下酒菜
创建实体 福内乌冬
创建实体 巡林官精选
创建实体 夏祭游鱼
创建实体 万民堂水煮鱼
创建实体 云遮玉
创建实体 伍玖叁式营养餐
创建实体 江湖百味
创建实体 关怀备至
创建实体 红炉一点雪」
创建实体 绝境求生烤鱼
创建实体 极致一钓
创建实体 蒙德往事」
创建实体 Null
创建实体 提瓦特焦蛋
创建实体 真味茶泡饭
创建实体 鸡！
创建实体 厚云朵松饼
创建实体 自有方圆」
创建实体 幽幽大行军
创建实体 侦察骑士烤肉！
创建实体 林之梦」
创建实体 魔法肉酱面
创建实体 爪爪土豆饼
创建实体 果香串烤
创建实体 饱腹感凝胶
创建实体 文火慢炖腌笃鲜
创建实体 哈瓦玛玛兹
创建实体 奇策」
创建实体 辣味时蔬烩肉
创建实体 乾坤摩拉肉
创建实体 强者之道
创建实体 鱼香吐司
创建实体 提神醒脑披萨
创建实体 史莱姆饮品
创建实体 决斗之魂
创建实体 婆娑一舞
创建实体 沾露虾仁
创建实体 没有未来菜
创建实体 憧憬
创建实体 永恒的信仰
创建实体 摩拉急速来
创建实体 盛世太平
创建实体 雨奇晴好
创建实体 安眠奢想
创建实体 静寂闲雅」
创建实体 常胜传说
创建实体 唯一的真相
创建实体 美梦」
创建实体 连心面
创建实体 暖意」
创建实体 古法秘制椰炭饼
创建实体 骇浪派
创建实体 祝圣交响乐
创建实体 炝炒肉片
创建实体 审判的晚宴
创建实体 蛋包饭圆舞曲
创建实体 风神杂烩菜
创建实体 生活）
创建实体 山珍凉卤面
创建实体 该角色无法参与料理


# 创建知识图谱关系（边）

In [20]:
def create_relationship(start_node, end_node, edges, rel_type, rel_name):
    '''创建关系函数'''
    for edge in edges:
        p = edge[0]
        q = edge[1]
        # 创建关系的 Cypher 语句
        query = "match(p:%s),(q:%s) where p.name='%s' and q.name='%s' create (p)-[rel:%s{name:'%s'}]->(q)" % (start_node, end_node, p, q, rel_type, rel_name)
        try:
            g.run(query) # 运行 Cypher 语句
            print('创建关系 {}-{}->{}'.format(p, rel_type, q))
        except Exception as e:
            print(e)

In [25]:
create_relationship('角色', 'City', rels_city, 'rels_city', '归属故里')
create_relationship('角色', 'Element', rels_element, 'rels_element', '技能性质')
create_relationship('角色', 'Weapon', rels_weapon, 'rels_weapon', '使用武器')
create_relationship('角色', 'Cook', rels_cook, 'Cook', '拿手好菜')
create_relationship('City', 'Element', city_element, 'city_element', '一方水土养一方人')

创建关系 琴-rels_city->蒙德
创建关系 迪卢克-rels_city->蒙德
创建关系 温迪-rels_city->蒙德
创建关系 可莉-rels_city->蒙德
创建关系 魈-rels_city->璃月
创建关系 旅行者-rels_city->其它
创建关系 派蒙-rels_city->其它
创建关系 莫娜-rels_city->蒙德
创建关系 刻晴-rels_city->璃月
创建关系 七七-rels_city->璃月
创建关系 钟离-rels_city->璃月
创建关系 达达利亚-rels_city->至冬
创建关系 阿贝多-rels_city->蒙德
创建关系 甘雨-rels_city->璃月
创建关系 神里绫华-rels_city->稻妻
创建关系 胡桃-rels_city->璃月
创建关系 雷电将军-rels_city->稻妻
创建关系 优菈-rels_city->蒙德
创建关系 宵宫-rels_city->稻妻
创建关系 枫原万叶-rels_city->稻妻
创建关系 八重神子-rels_city->稻妻
创建关系 珊瑚宫心海-rels_city->稻妻
创建关系 埃洛伊-rels_city->其它
创建关系 神里绫人-rels_city->稻妻
创建关系 荒泷一斗-rels_city->稻妻
创建关系 赛诺-rels_city->须弥
创建关系 申鹤-rels_city->璃月
创建关系 夜兰-rels_city->璃月
创建关系 提纳里-rels_city->须弥
创建关系 妮露-rels_city->须弥
创建关系 纳西妲-rels_city->须弥
创建关系 流浪者-rels_city->须弥
创建关系 北斗-rels_city->璃月
创建关系 安柏-rels_city->蒙德
创建关系 丽莎-rels_city->蒙德
创建关系 凯亚-rels_city->蒙德
创建关系 芭芭拉-rels_city->蒙德
创建关系 雷泽-rels_city->蒙德
创建关系 行秋-rels_city->璃月
创建关系 菲谢尔-rels_city->蒙德
创建关系 诺艾尔-rels_city->蒙德
创建关系 班尼特-rels_city->蒙德
创建关系 重云-rels_city->璃月
创建关系 香菱-rels_city->璃月
创建关系 凝